In [1]:
import geopandas as gpd
import os

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from utils import CensusDataLoader, ERSDataLoader

load_dotenv()

True

In [2]:
acs_loader = CensusDataLoader(api_key=os.getenv("CENSUS_API_KEY"))
ers_loader = ERSDataLoader(continental=True)

Census API key has been set for this session.
Getting data from the 2018-2022 5-year ACS


In [3]:
years = [2013, 2018, 2023]
dfs = []
for yr in years:
    df = pd.read_csv(f"ice-detention-trends/facilities/by_fiscal_year/FY{yr}.csv")
    df["fiscal_year"] = yr
    dfs.append(df)
    
detention_data = pd.concat(dfs, ignore_index=True)

# Group by facility and year
detention_data = detention_data.groupby(["detention_facility_code", "fiscal_year"]).agg({
    "daily_pop": "sum",
    "midnight_pop": "sum"
}).reset_index()

# Load metadata and merge
facilities = pd.read_csv("ice-detention-trends/metadata/facilities.csv")
detention_data = detention_data.merge(
    facilities[["detention_facility_code", "detention_facility_name", "latitude", "longitude", "state", "county"]], 
    on="detention_facility_code", 
    how="left"
)

detention_data.head()

,detention_facility_code,fiscal_year,daily_pop,midnight_pop,detention_facility_name,latitude,longitude,state,county
0,AAORDMD,2013,0,0,Ordnance Road Correctional Center,39.199609,-76.594162,MD,Anne Arundel
1,AAORDMD,2018,27195,26818,Ordnance Road Correctional Center,39.199609,-76.594162,MD,Anne Arundel
2,AAORDMD,2023,0,0,Ordnance Road Correctional Center,39.199609,-76.594162,MD,Anne Arundel
3,ABIRJVA,2013,0,0,B.R.R.J. Abingdon,36.732440,-81.911639,VA,Washington
4,ABIRJVA,2018,0,0,B.R.R.J. Abingdon,36.732440,-81.911639,VA,Washington


In [ ]:
# Create geometry point column
# TODO: instead of applying to ice_data, apply this to facilities instead.
ice_data = gpd.GeoDataFrame(
    detention_data, 
    geometry=gpd.points_from_xy(detention_data.longitude, detention_data.latitude),
    crs="EPSG:4326"
)

ice_data.head()

,detention_facility_code,fiscal_year,daily_pop,midnight_pop,detention_facility_name,latitude,longitude,state,county,geometry
0,AAORDMD,2013,0,0,Ordnance Road Correctional Center,39.199609,-76.594162,MD,Anne Arundel,POINT (-76.59416 39.19961)
1,AAORDMD,2018,27195,26818,Ordnance Road Correctional Center,39.199609,-76.594162,MD,Anne Arundel,POINT (-76.59416 39.19961)
2,AAORDMD,2023,0,0,Ordnance Road Correctional Center,39.199609,-76.594162,MD,Anne Arundel,POINT (-76.59416 39.19961)
3,ABIRJVA,2013,0,0,B.R.R.J. Abingdon,36.732440,-81.911639,VA,Washington,POINT (-81.91164 36.73244)
4,ABIRJVA,2018,0,0,B.R.R.J. Abingdon,36.732440,-81.911639,VA,Washington,POINT (-81.91164 36.73244)


In [5]:
MOBILITY_VARS = {
    "MOBILITY_TOTAL": "B07012_002E",
    "SAME_HOUSE": "B07012_006E",
    "MOVED_SAME_COUNTY": "B07012_010E",
    "MOVED_DIFF_COUNTY": "B07012_014E",
    "MOVED_DIFF_STATE": "B07012_018E",
    "MOVED_ABROAD": "B07012_022E",
}
SNAP_VARS = {"SNAP_TOTAL": "B22003_001E", "RECEIVING": "B22003_002E"}
EMPLOYMENT_VARS = {"LABOR_FORCE": "B23025_002E", "UNEMPLOYED": "B23025_005E"}
EDUCATION_VARS = {
    "EDU_TOTAL": "B15003_001E",
    "GRADES": [f"B15003_{num:03d}E" for num in range(2, 17)],
}
QUALITY_VARS = {
    "PLUMB_TOTAL": "B25047_001E",
    "LACK_PLUMBING": "B25047_003E",
    "KITCH_TOTAL": "B25051_001E",
    "LACK_KITCHEN": "B25051_003E",
}
POVERTY_VARS = {"POV_TOTAL": "B17001_001E", "POVERTY": "B17001_002E"}
MORTGAGE_VARS = {
    "MORT_TOTAL": "B25091_001E",
    "MORT_30_35": "B25091_008E",
    "MORT_35_40": "B25091_009E",
    "MORT_40_50": "B25091_010E",
    "MORT_50_PLUS": "B25091_011E",
    "NO_MORT_30_35": "B25091_019E",
    "NO_MORT_35_40": "B25091_020E",
    "NO_MORT_40_50": "B25091_021E",
    "NO_MORT_50_PLUS": "B25091_022E",
}
RENT_VARS = {
    "RENT_TOTAL": "B25070_001E",
    "RENT_30_35": "B25070_007E",
    "RENT_35_40": "B25070_008E",
    "RENT_40_50": "B25070_009E",
    "RENT_50_PLUS": "B25070_010E",
}
MISC_VARS = {"GINI": "B19083_001E"}

all_vars_dict = {}
for d in [
    MOBILITY_VARS,
    SNAP_VARS,
    EMPLOYMENT_VARS,
    EDUCATION_VARS,
    QUALITY_VARS,
    POVERTY_VARS,
    MORTGAGE_VARS,
    RENT_VARS,
    MISC_VARS,
]:
    for key, val in d.items():
        if isinstance(val, list):
            for i, item in enumerate(val):
                all_vars_dict[f"{key}_{i}"] = item
        else:
            all_vars_dict[key] = val

In [6]:
# Fetch ACS data for 2013, 2018, 2023 (with caching)
CACHE_PATH = ".data/RSS_ACS_DATA.csv"

if os.path.exists(CACHE_PATH):
    print("Loading cached ACS data from .data/RSS_ACS_DATA.csv")
    acs_data = pd.read_csv(CACHE_PATH)
else:
    print("Fetching ACS data from Census API...")
    acs_data = acs_loader.fetch_multiple_years(
        years=[2013, 2018, 2023], 
        variables=all_vars_dict,
        geometry='county'
    )
    os.makedirs(".data", exist_ok=True)
    acs_data.to_csv(CACHE_PATH, index=False)
    print(f"Saved ACS data to {CACHE_PATH}")

acs_data["GEOID"] = pd.to_numeric(acs_data["GEOID"], errors="coerce").astype("Int64")
acs_data.head()

Loading cached ACS data from .data/RSS_ACS_DATA.csv


,GEOID,MOBILITY_TOTAL,SAME_HOUSE,MOVED_SAME_COUNTY,MOVED_DIFF_COUNTY,MOVED_DIFF_STATE,MOVED_ABROAD,SNAP_TOTAL,RECEIVING,LABOR_FORCE,...,NO_MORT_35_40,NO_MORT_40_50,NO_MORT_50_PLUS,RENT_TOTAL,RENT_30_35,RENT_35_40,RENT_40_50,RENT_50_PLUS,GINI,year
0,10001,20010.0,15786.0,2922.0,413.0,720.0,169.0,58524,8782,82341.0,...,324.0,318.0,625.0,16190.0,1443.0,1227.0,1584.0,4012.0,0.4046,2013
1,10003,55595.0,38817.0,11212.0,516.0,4436.0,614.0,200739,21671,288705.0,...,858.0,820.0,1973.0,60206.0,5671.0,3986.0,5354.0,14684.0,0.4475,2013
2,10005,25936.0,21249.0,2799.0,217.0,1090.0,581.0,76444,10137,95650.0,...,831.0,748.0,1706.0,16088.0,1309.0,1216.0,1325.0,3237.0,0.4275,2013
3,9001,81796.0,63581.0,12524.0,1232.0,2523.0,1936.0,332655,27063,498574.0,...,2622.0,3628.0,8856.0,103169.0,8511.0,6148.0,9505.0,28335.0,0.5391,2013
4,9003,98911.0,74317.0,17314.0,2781.0,2551.0,1948.0,347874,45234,486537.0,...,2199.0,3316.0,5761.0,119920.0,9547.0,7526.0,9427.0,31777.0,0.4682,2013


In [ ]:
# Fetch RUCC codes for urban/rural classification
rucc_data = ers_loader.collect_rucc_data()
# rucc_data["GEOID"] = rucc_data["GEOID"].astype(int)
rucc_data.head()

Attribute,GEOID,State,County_Name,Description,Population_2020,RUCC
0,01001,AL,autauga,"Metro - Counties in metro areas of 250,000 to ...",58805,2
1,01003,AL,baldwin,Metro - Counties in metro areas of fewer than ...,231767,3
2,01005,AL,barbour,"Nonmetro - Urban population of 5,000 to 20,000...",25223,6
3,01007,AL,bibb,Metro - Counties in metro areas of 1 million p...,22293,1
4,01009,AL,blount,Metro - Counties in metro areas of 1 million p...,59134,1


In [8]:
# Load county geometry for spatial merge
county_geo = acs_loader.collect_geometry_data(geometry_type="counties", year=2022)
county_geo["GEOID"] = pd.to_numeric(county_geo["GEOID"], errors="coerce").astype("Int64")
print(f"Loaded {len(county_geo)} counties")
county_geo.head()

Loaded 3222 counties


,STATEFP,COUNTYFP,COUNTYNS,AFFGEOID,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,01,069,00161560,0500000US01069,1069,Houston,Houston County,AL,Alabama,06,1501742235,4795415,"POLYGON ((-85.71209 31.19727, -85.70934 31.198..."
1,01,023,00161537,0500000US01023,1023,Choctaw,Choctaw County,AL,Alabama,06,2365900083,19114321,"POLYGON ((-88.47323 31.89386, -88.46888 31.930..."
2,01,005,00161528,0500000US01005,1005,Barbour,Barbour County,AL,Alabama,06,2292160151,50523213,"POLYGON ((-85.74803 31.61918, -85.74544 31.618..."
3,01,107,00161580,0500000US01107,1107,Pickens,Pickens County,AL,Alabama,06,2282835044,22621093,"POLYGON ((-88.34043 32.9912, -88.33101 33.0724..."
4,01,033,00161542,0500000US01033,1033,Colbert,Colbert County,AL,Alabama,06,1535742270,79160396,"POLYGON ((-88.13925 34.5878, -88.13872 34.5892..."


In [9]:
# Filter county columns (keep names) and merge with ACS
# NAMELSAD = County Name (e.g. "Autauga County"), STATE_NAME = State Name
county_cols = ["GEOID", "NAMELSAD", "STATE_NAME", "geometry"]
acs_gdf = county_geo[county_cols].merge(acs_data, on="GEOID", how="inner")
acs_gdf.head()

,GEOID,NAMELSAD,STATE_NAME,geometry,MOBILITY_TOTAL,SAME_HOUSE,MOVED_SAME_COUNTY,MOVED_DIFF_COUNTY,MOVED_DIFF_STATE,MOVED_ABROAD,...,NO_MORT_35_40,NO_MORT_40_50,NO_MORT_50_PLUS,RENT_TOTAL,RENT_30_35,RENT_35_40,RENT_40_50,RENT_50_PLUS,GINI,year
0,1069,Houston County,Alabama,"POLYGON ((-85.71209 31.19727, -85.70934 31.198...",17823.0,13643.0,2560.0,895.0,689.0,36.0,...,183.0,184.0,333.0,13881.0,1058.0,797.0,1088.0,3027.0,0.4745,2013
1,1069,Houston County,Alabama,"POLYGON ((-85.71209 31.19727, -85.70934 31.198...",18784.0,16339.0,1328.0,514.0,477.0,126.0,...,145.0,157.0,505.0,15365.0,1222.0,801.0,1104.0,3276.0,0.4788,2023
2,1023,Choctaw County,Alabama,"POLYGON ((-88.47323 31.89386, -88.46888 31.930...",2880.0,2474.0,266.0,29.0,111.0,0.0,...,44.0,116.0,192.0,898.0,69.0,50.0,38.0,141.0,0.4814,2013
3,1023,Choctaw County,Alabama,"POLYGON ((-88.47323 31.89386, -88.46888 31.930...",2380.0,2303.0,32.0,26.0,17.0,2.0,...,76.0,99.0,150.0,920.0,59.0,0.0,34.0,184.0,0.4554,2023
4,1005,Barbour County,Alabama,"POLYGON ((-85.74803 31.61918, -85.74544 31.618...",6365.0,5326.0,686.0,53.0,206.0,94.0,...,141.0,128.0,191.0,2973.0,154.0,149.0,193.0,693.0,0.4658,2013


In [ ]:
# TODO: instead of doing this weird per year spatial join thing, just do a spatial join between the acs geometry and the facilities polygons after the previous TODO is implemented. just join it so that there can be multiple facilities per county. then the 
# Perform spatial join and select final columns
joined_dfs = []
for yr in [2013, 2018, 2023]:
    detention_yr = ice_data[ice_data["fiscal_year"] == yr]
    acs_yr = acs_gdf[acs_gdf["year"] == yr]
    
    if not detention_yr.empty and not acs_yr.empty:
        joined = gpd.sjoin(detention_yr, acs_yr, how="inner", predicate="within")
        joined_dfs.append(joined)

detention_final = pd.concat(joined_dfs, ignore_index=True)

detention_final["GEOID"] = detention_final["GEOID"].astype(int)
rucc_data["GEOID"] = rucc_data["GEOID"].astype(int)



# Merge RUCC data
detention_final = detention_final.merge(rucc_data[["GEOID", "RUCC"]], on="GEOID", how="left")

# Final column selection including RUCC
keep_cols = [
    "detention_facility_code", "detention_facility_name", "fiscal_year",
    "STATE_NAME", "NAMELSAD", "GEOID", "RUCC",
    "daily_pop", "midnight_pop", "latitude", "longitude", "geometry",
    "RiskyMobility", "SnapRate", "Unemployment", "No_HS",
    "HousingQuality", "Poverty", "CostBurden"
]
detention_final = detention_final[keep_cols]
detention_final = detention_final.rename(columns={"NAMELSAD": "county_name", "STATE_NAME": "state_name", "RUCC": "rucc_code"})

detention_final.head()

KeyError: "['RiskyMobility', 'SnapRate', 'Unemployment', 'No_HS', 'HousingQuality', 'Poverty', 'CostBurden'] not in index"

In [ ]:
# Distribution of detention populations (per year, limit outliers)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, yr in enumerate([2013, 2018, 2023]):
    sub = detention_final[detention_final["fiscal_year"] == yr]
    # Trim to below 95th percentile for readability
    cap = sub["daily_pop"].quantile(0.95)
    sub = sub[sub["daily_pop"] <= cap]
    axes[i].hist(sub["daily_pop"], bins=40, edgecolor="white")
    axes[i].set_title(f"{yr} (capped @ {cap:.0f})")
    axes[i].set_xlabel("Daily Population")
    axes[i].set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Yearly trends: total and average detention population
yearly = detention_final.groupby("fiscal_year").agg(
    total_daily=("daily_pop", "sum"),
    avg_daily=("daily_pop", "mean"),
    n_facilities=("detention_facility_code", "nunique")
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(yearly["fiscal_year"].astype(str), yearly["total_daily"], color="steelblue")
axes[0].set_title("Total Daily Population")
axes[0].set_xlabel("Year")

axes[1].bar(yearly["fiscal_year"].astype(str), yearly["avg_daily"], color="coral")
axes[1].set_title("Avg Daily Population per Facility")
axes[1].set_xlabel("Year")

axes[2].bar(yearly["fiscal_year"].astype(str), yearly["n_facilities"], color="seagreen")
axes[2].set_title("Number of Facilities")
axes[2].set_xlabel("Year")

plt.tight_layout()
plt.show()

In [ ]:
# Detention by Urban/Rural (RUCC)
# Group RUCC into Metro (1-3) vs Nonmetro (4-9)
detention_final["urban_type"] = detention_final["rucc_code"].apply(
    lambda x: "Metro" if pd.notna(x) and x <= 3 else ("Nonmetro" if pd.notna(x) else "Unknown")
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Total pop by urban type per year
urb_year = detention_final.groupby(["fiscal_year", "urban_type"])["daily_pop"].sum().reset_index()
sns.barplot(data=urb_year, x="fiscal_year", y="daily_pop", hue="urban_type", ax=axes[0])
axes[0].set_title("Total Detention by Urban/Rural Type")
axes[0].set_xlabel("Year")

# Demographics by urban type
dem_cols = ["Poverty", "Unemployment", "No_HS", "CostBurden"]
dem_urb = detention_final.groupby("urban_type")[dem_cols].mean().T
dem_urb.plot(kind="bar", ax=axes[1])
axes[1].set_title("Avg Demographics by Urban/Rural Type")
axes[1].set_ylabel("Rate")
axes[1].legend(title="RuCC Type")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of key variables
corr_cols = ["daily_pop", "RiskyMobility", "SnapRate", "Unemployment", 
             "No_HS", "HousingQuality", "Poverty", "CostBurden"]
corr = detention_final[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation: Detention Pop vs Demographics")
plt.tight_layout()
plt.show()

In [ ]:
# Top facilities by population
top_fac = detention_final.groupby(["detention_facility_name", "fiscal_year"])["daily_pop"].sum().reset_index()
top_fac = top_fac.sort_values("daily_pop", ascending=False).head(20)

plt.figure(figsize=(14, 6))
sns.barplot(data=top_fac, y="detention_facility_name", x="daily_pop", hue="fiscal_year", dodge=False)
plt.title("Top 20 Facilities by Daily Population")
plt.xlabel("Total Daily Population")
plt.ylabel("")
plt.tight_layout()
plt.show()